# Fault-code image generator

Enter a **fault code** + **machine** and generate a fresh set of images.

## Where do training images come from?

We have **no historical customer claim photos** in this repo.  
Training for GAN/diffusion is **100% simulated** by `ml/fault_code_vision/panel_render.py`:

| Simulated factor | Real claim scenario |
|------------------|---------------------|
| Room backdrop | Laundry, kitchen, tile, wood, garage, window light |
| Appliance finish | White, stainless, black, graphite, beige |
| Display type | Green/blue LCD, gray LCD + black digits, LED, OLED |
| Framing | Tight LCD, full control panel, machine-in-room |
| Phone augments | Tilt, blur, glare, noise, JPEG |

**Learning rule:** models only generate what they saw in training.  
Old checkpoints trained on dark-only panels will still look dark — delete them to retrain.

| Method | What you get |
|--------|----------------|
| `procedural` | Sharp **diverse** panels/scenes (always readable) |
| `phone` | Claim-photo realism on those scenes |
| `gan` | cGAN samples (retrain on diverse set if ckpt is old) |
| `diffusion` | DDPM samples (needs 50+ epochs; retrain for diversity) |

**Tip:** For client demos of readable codes, use `procedural` + `phone`. Force retrain: `rm -rf notebooks/fault_code_gen_artifacts/cgan notebooks/fault_code_gen_artifacts/diffusion`


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path(".").resolve()
if not (ROOT / "ml").is_dir() and (ROOT.parent / "ml").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ml.fault_code_vision.generate import generate_images
from ml.fault_code_vision.catalog import catalog_codes

print("Known codes:", ", ".join(catalog_codes()))
print("Machines: washer | dryer | dishwasher | fridge | generic")

from ml.fault_code_vision.panel_render import describe_data_provenance
import json
print("\nData provenance:")
print(json.dumps(describe_data_provenance(), indent=2))

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


## Configure request

Edit the cell below, then run it.

In [ ]:
# === INPUTS (edit these) ===
FAULT_CODE = "5E"          # e.g. UE, 5E, E24, F9E1
MACHINE = "washer"         # washer | dryer | dishwasher | fridge | generic
N = 4                      # images per method
METHODS = ["procedural", "phone", "gan", "diffusion"]

# Training if checkpoints missing (diffusion needs more epochs to leave pure noise)
AUTO_TRAIN = True
GAN_EPOCHS = 15            # raise to 30–50 for better GAN digits
DIFFUSION_EPOCHS = 50      # raise to 60–100 on CUDA for cleaner diffusion

OUT = ROOT / "notebooks" / "fault_code_gen_artifacts" / "requests"

meta = generate_images(
    fault_code=FAULT_CODE,
    machine=MACHINE,
    n=N,
    methods=METHODS,
    out_dir=OUT,
    size=256,
    auto_train=AUTO_TRAIN,
    gan_epochs=GAN_EPOCHS,
    diffusion_epochs=DIFFUSION_EPOCHS,
)
print("Wrote to:", meta["out_dir"])

In [ ]:
# Preview everything that was generated
for method, paths in meta["outputs"].items():
    fig, axes = plt.subplots(1, len(paths), figsize=(3 * len(paths), 3))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(Image.open(p))
        ax.set_title(Path(p).name, fontsize=8)
        ax.axis("off")
    fig.suptitle(f"{method} — {meta['machine']} / {meta['fault_code']}")
    plt.tight_layout()
    plt.show()

cs = Path(meta["out_dir"]) / "contact_sheet.png"
if cs.exists():
    print("Contact sheet:", cs)
    try:
        from IPython.display import display as _disp
    except ImportError:
        _disp = lambda x: None
    _disp(Image.open(cs))
    # also show with matplotlib for plain kernels
    plt.figure(figsize=(8, 4))
    plt.imshow(Image.open(cs))
    plt.axis("off")
    plt.title("contact sheet")
    plt.show()

## CLI equivalent

```bash
python -m ml.fault_code_vision.generate --code 5E --machine washer --n 6 \
  --methods procedural,phone,gan,diffusion \
  --diffusion-epochs 50 --gan-epochs 20
```

Sharp demos only (no train wait):

```bash
python -m ml.fault_code_vision.generate --code UE --machine washer --methods procedural,phone --n 8
```